# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, examine, and process the FAIR² colorectal cancer survivors dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/api/#python-api) library, which provides tools for working with Croissant data packages.

### Dataset Source
The dataset is described by a Croissant schema accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant library (if not already installed). Remove the exclamation mark if running outside Jupyter.
!pip install mlcroissant

## 1. Data Loading

Load the Croissant metadata and underlying records for the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata as an object
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Display all available record sets, their `@id` fields, and key field or column ids. Entities in the schema (record sets, fields, columns) are referenced using their `@id` values.

In [ ]:
# List all record sets and their fields using their @id
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"- Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Fields (by @id):")
    for field in rs.fields:
        print(f"    - {field.name}: {field.id}")
    print(f"  Columns (by @id):")
    for col in getattr(rs, 'columns', []):
        print(f"    - {col.name}: {col.id}")
    print()

## 3. Data Extraction

We will extract the records from all available record sets in the dataset and load them into pandas DataFrames. Use `@id` fields for record set references.

In [ ]:
# Collect @id values for available record sets
record_set_ids = [rs.id for rs in record_sets]
print("Record sets available for extraction:")
for rid in record_set_ids:
    print(f"- {rid}")

# Extract all records from each record set as pandas DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# For this dataset, let's print available columns for the first record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id is not None:
    print(f"\nColumns for record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print('No record sets found in dataset.')

## 4. Exploratory Data Analysis (EDA)

Let us demonstrate typical processing: filtering, normalization, and grouping by a field. Operations use `@id` to reference fields, consistent with Croissant conventions.

For this dataset, let's suppose we have a field for patient `@id` age (`http://mlcommons.org/croissant/field/age`) and for `Sex` (`http://mlcommons.org/croissant/field/sex`). Replace these with correct `@id` values as needed based on actual output from section 2.

In [ ]:
# Set field @ids (ensure these match those printed in the Data Overview)
# Use the actual @id for age and sex from your output above.
# If unsure, run the Data Overview cell and inspect output to set these values.

main_df = dataframes[main_record_set_id]

# Example: Use @id for age and sex. Replace with IDs from your data.
# E.g. age_field_id = 'http://mlcommons.org/croissant/field/age'  # replace
# sex_field_id = 'http://mlcommons.org/croissant/field/sex'      # replace

age_field_candidates = [col for col in main_df.columns if 'age' in col.lower()]
sex_field_candidates = [col for col in main_df.columns if 'sex' in col.lower() or 'gender' in col.lower()]

if age_field_candidates:
    numeric_field = age_field_candidates[0]
    print(f"Using field @id for age: {numeric_field}")
else:
    print("No age field found.")

if sex_field_candidates:
    group_field = sex_field_candidates[0]
    print(f"Using field @id for sex: {group_field}")
else:
    group_field = None
    print("No sex field found.")

# Filtering: Show records with age > 60 (as an example; adjust threshold as needed)
if age_field_candidates:
    threshold = 60
    df_num = pd.to_numeric(main_df[numeric_field], errors='coerce')
    filtered_df = main_df[df_num > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalization (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (df_num - df_num.mean()) / df_num.std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping by sex/gender field
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        display(grouped_df)
else:
    print("No numeric (age) field found for EDA.")

## 5. Visualization

Let's visualize the distribution and demographic comparison based on the fields analyzed above. We use matplotlib for plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if age (numeric_field) is available:
if age_field_candidates:
    plt.figure(figsize=(8, 5))
    sns.histplot(pd.to_numeric(main_df[numeric_field], errors='coerce').dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel("Age")
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group (e.g. Sex)
    if group_field:
        plt.figure(figsize=(7, 5))
        sns.boxplot(x=main_df[group_field], y=pd.to_numeric(main_df[numeric_field], errors='coerce'))
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel("Sex")
        plt.ylabel("Age")
        plt.show()
else:
    print("No numeric field found to plot.")

## 6. Conclusion

This notebook demonstrated how to use the `mlcroissant` library to explore a clinical dataset described by a Croissant schema. We identified available record sets and fields by their `@id`, extracted the data, performed basic cleaning and normalization, grouped by demographic fields, and visualized field distributions.

Further analysis can be carried out by referencing additional fields via their `@id` as needed, leveraging the Croissant metadata for reproducible data science workflows. For more, see the [Croissant Python API](https://mlcommons.github.io/croissant/api/#python-api) documentation.